# 4 - Collect and prepare weather data

Collect airport weather observations, and check coverage and quality of those observations. Match observations to each flight's prediction cutoff.

In [30]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent  

NOAA_DIR = PROJECT_ROOT / "data/raw/noaa"

weather = pd.read_csv(NOAA_DIR / "2024/noaa_atl_2024.psv", sep="|", low_memory=False)
boundary = pd.read_csv(NOAA_DIR / "2025/noaa_atl_2025.psv", sep="|", low_memory=False)

dates = pd.to_datetime(boundary["DATE"], utc=True)
boundary = boundary.loc[dates.ge("2025-01-01") & dates.lt("2025-01-02")]

assert not boundary.empty
assert set(boundary["STATION"]) == set(weather["STATION"])

weather = pd.concat([weather, boundary], ignore_index=True)

print(f"{len(weather)} rows, {len(weather.columns)} cols")
print(weather.columns.tolist())
display(weather.head())

13212 rows, 329 cols
['STATION', 'Station_name', 'DATE', 'Year', 'Month', 'Day', 'Hour', 'Minute', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'temperature', 'temperature_Measurement_Code', 'temperature_Quality_Code', 'temperature_Report_Type', 'temperature_Source_Code', 'temperature_Source_Station_ID', 'dew_point_temperature', 'dew_point_temperature_Measurement_Code', 'dew_point_temperature_Quality_Code', 'dew_point_temperature_Report_Type', 'dew_point_temperature_Source_Code', 'dew_point_temperature_Source_Station_ID', 'station_level_pressure', 'station_level_pressure_Measurement_Code', 'station_level_pressure_Quality_Code', 'station_level_pressure_Report_Type', 'station_level_pressure_Source_Code', 'station_level_pressure_Source_Station_ID', 'sea_level_pressure', 'sea_level_pressure_Measurement_Code', 'sea_level_pressure_Quality_Code', 'sea_level_pressure_Report_Type', 'sea_level_pressure_Source_Code', 'sea_level_pressure_Source_Station_ID', 'wind_direction', 'wind_direction_Measurement_C

,STATION,Station_name,DATE,Year,Month,Day,Hour,Minute,LATITUDE,LONGITUDE,...,precipitation_24_hour_Quality_Code,precipitation_24_hour_Report_Type,precipitation_24_hour_Source_Code,precipitation_24_hour_Source_Station_ID,REM,REM_Measurement_Code,REM_Quality_Code,REM_Report_Type,REM_Source_Code,REM_Source_Station_ID
0,USW00013874,ATLANTA HARTSFIELD-JACKSON INT,2024-01-01T00:00:00,2024,1,1,0,0,33.6297,-84.4422,...,NaN,NaN,NaN,NaN,SYN08072219 32966 42204 10094 20000 39825 4020...,NaN,NaN,FM12,223.0,ICAO-KATL
1,USW00013874,ATLANTA HARTSFIELD-JACKSON INT,2024-01-01T00:52:00,2024,1,1,0,52,33.6297,-84.4422,...,NaN,NaN,NaN,NaN,MET09912/31/23 19:52:03 METAR KATL 010052Z 220...,NaN,NaN,FM15,343.0,722190-13874
2,USW00013874,ATLANTA HARTSFIELD-JACKSON INT,2024-01-01T01:52:00,2024,1,1,1,52,33.6297,-84.4422,...,NaN,NaN,NaN,NaN,MET09912/31/23 20:52:03 METAR KATL 010152Z 250...,NaN,NaN,FM15,343.0,722190-13874
3,USW00013874,ATLANTA HARTSFIELD-JACKSON INT,2024-01-01T02:52:00,2024,1,1,2,52,33.6297,-84.4422,...,NaN,NaN,NaN,NaN,MET11212/31/23 21:52:03 METAR KATL 010252Z 230...,NaN,NaN,FM15,343.0,722190-13874
4,USW00013874,ATLANTA HARTSFIELD-JACKSON INT,2024-01-01T03:00:00,2024,1,1,3,0,33.6297,-84.4422,...,NaN,NaN,NaN,NaN,SYN06472219 32966 62305 10078 20000 39821 4020...,NaN,NaN,FM12,223.0,ICAO-KATL


In [31]:
display(weather[
    ["STATION", "Station_name", "LATITUDE", "LONGITUDE"]
].drop_duplicates())

dates = pd.to_datetime(weather["DATE"], errors="coerce")

print("Date range:", dates.min(), "to", dates.max())
print("Invalid dates:", dates.isna().sum())
print("Repeated station/timestamps:",
      weather.duplicated(["STATION", "DATE"]).sum())

,STATION,Station_name,LATITUDE,LONGITUDE
0,USW00013874,ATLANTA HARTSFIELD-JACKSON INT,33.6297,-84.4422


Date range: 2024-01-01 00:00:00 to 2025-01-01 23:52:00
Invalid dates: 0
Repeated station/timestamps: 0


In [32]:
weather_vars = ["temperature", "dew_point_temperature", "wind_speed", "wind_gust", "visibility", "ceiling_height", "precipitation"]

display(weather[weather_vars].isna().mean().mul(100).round(1).rename("missing_pct").to_frame())

for variable in weather_vars:
    print(f"{variable}:")
    print(weather[f"{variable}_Quality_Code"].value_counts(dropna=False))

,missing_pct
temperature,0.0
dew_point_temperature,0.0
wind_speed,0.0
wind_gust,87.9
visibility,0.0
ceiling_height,14.5
precipitation,29.2


temperature:
temperature_Quality_Code
5      10159
1       3005
5.0       24
A         15
1.0        8
NaN        1
Name: count, dtype: int64
dew_point_temperature:
dew_point_temperature_Quality_Code
5      10161
1       3005
5.0       24
A         13
1.0        8
NaN        1
Name: count, dtype: int64
wind_speed:
wind_speed_Quality_Code
5      10204
1       2973
5.0       24
1.0        8
NaN        2
U          1
Name: count, dtype: int64
wind_gust:
wind_gust_Quality_Code
NaN    11615
5.0     1581
1.0       16
Name: count, dtype: int64
visibility:
visibility_Quality_Code
5      10068
1       2973
A         81
6         58
5.0       24
1.0        8
Name: count, dtype: int64
ceiling_height:
ceiling_height_Quality_Code
5.0    10171
NaN     1914
1.0     1042
4.0       58
6.0       27
Name: count, dtype: int64
precipitation:
precipitation_Quality_Code
5.0    8695
NaN    3853
1.0     664
Name: count, dtype: int64


In [33]:
for variable in weather_vars:
    print(f"{variable}:")
    display(weather[[
        f"{variable}_Source_Code",
        f"{variable}_Quality_Code",
    ]].value_counts(dropna=False).rename("rows").to_frame())

temperature:


rows
temperature_Source_Code temperature_Quality_Code       
343.0                   5                         10159
223.0                   1                          3005
343.0                   5.0                          24
                        A                            15
223.0                   1.0                           8
NaN                     NaN                           1

dew_point_temperature:


rows
dew_point_temperature_Source_Code dew_point_temperature_Quality_Code       
343.0                             5                                   10161
223.0                             1                                    3005
343.0                             5.0                                    24
                                  A                                      13
223.0                             1.0                                     8
NaN                               NaN                                     1

wind_speed:


,,rows
wind_speed_Source_Code,wind_speed_Quality_Code,
343.0,5,10204
223.0,1,2973
343.0,5.0,24
223.0,1.0,8
NaN,NaN,2
343.0,U,1


... (4 outputs left out)

ceiling_height:


rows
ceiling_height_Source_Code ceiling_height_Quality_Code       
343.0                      5.0                          10171
NaN                        NaN                           1914
223.0                      1.0                           1042
                           4.0                             58
343.0                      6.0                             27

precipitation:


,,rows
precipitation_Source_Code,precipitation_Quality_Code,
343.0,5.0,8695
NaN,NaN,3853
223.0,1.0,664


## Apply source-specific quality filters

In [34]:
weather_clean = weather.copy()
quality_audit = []

for variable in weather_vars:
    source = pd.to_numeric(weather[f"{variable}_Source_Code"])
    quality = weather[f"{variable}_Quality_Code"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    keep = (source.eq(343) & quality.isin(["1", "5", "A"])) | (source.eq(223) & quality.eq("1")).fillna(False)

    weather_clean[variable] = weather[variable].where(keep)
    quality_audit.append({
        "variable": variable,
        "newly_masked": (weather[variable].notna() & ~keep).sum(),
        "missing_pct": weather_clean[variable].isna().mean() * 100,
    })

display(pd.DataFrame(quality_audit).round(2))

,variable,newly_masked,missing_pct
0,temperature,0,0.01
1,dew_point_temperature,0,0.01
2,wind_speed,1,0.02
3,wind_gust,0,87.91
4,visibility,58,0.44
5,ceiling_height,85,15.13
6,precipitation,0,29.16


In [35]:
display(weather_clean[weather_vars].describe().T[
    ["count", "min", "50%", "max"]
])

for var in weather_vars:
    available = weather_clean[var].notna()
    print(f"{var}:")
    display(weather_clean.loc[available, [
        f"{var}_Source_Code",
        f"{var}_Measurement_Code",
    ]].value_counts(dropna=False).rename("rows").to_frame())

,count,min,50%,max
temperature,13211.0,-10.0,19.400,37.200
dew_point_temperature,13211.0,-16.7,13.900,25.000
wind_speed,13209.0,0.0,3.600,16.000
wind_gust,1597.0,7.2,10.800,25.200
visibility,13154.0,0.0,16.093,16.093
ceiling_height,11213.0,0.0,22000.000,22000.000
precipitation,9359.0,0.0,0.000,62.500


temperature:


,,rows
temperature_Source_Code,temperature_Measurement_Code,
343.0,NaN,10198
223.0,NaN,3013


dew_point_temperature:


,,rows
dew_point_temperature_Source_Code,dew_point_temperature_Measurement_Code,
343.0,NaN,10198
223.0,NaN,3013


wind_speed:


... (5 outputs left out)

ceiling_height:


rows
ceiling_height_Source_Code ceiling_height_Measurement_Code      
343.0                      M;N                              5501
                           ;N                               4634
223.0                      ;N                               1010
343.0                      W;N                                36
223.0                      W;N                                32

precipitation:


rows
precipitation_Source_Code precipitation_Measurement_Code      
343.0                     NaN                             8241
                          T                                454
223.0                     NaN                              445
                          T                                219

Review two specific cases:
- preserve trace precipitation
- use unlimited ceiling as a separated measurement

In [36]:
weather_clean["precipitation_trace"] = (
    weather_clean["precipitation_Measurement_Code"].eq("T").astype("boolean").where(weather_clean["precipitation"].notna())
)

weather_clean["ceiling_unlimited"] = (
    weather_clean["ceiling_height"].eq(22000).astype("boolean").where(weather_clean["ceiling_height"].notna())
)

weather_clean["ceiling_height"] = weather_clean["ceiling_height"].mask(
    weather_clean["ceiling_unlimited"].fillna(False)
)

## Prepare weahter timestamps

NOAA timestamps is in UTC, so no need to convert

In [37]:
weather_clean["weather_time_utc"] = pd.to_datetime(weather_clean["DATE"], utc=True, errors="raise")
weather_clean = weather_clean.sort_values("weather_time_utc")

assert weather_clean["STATION"].nunique() == 1
assert weather_clean["weather_time_utc"].notna().all()
assert not weather_clean["weather_time_utc"].duplicated().any()

gaps = weather_clean["weather_time_utc"].diff().dt.total_seconds() / 60

print("Coverage:", weather_clean["weather_time_utc"].min(), "to", weather_clean["weather_time_utc"].max())
print("Median gap (mins):", gaps.median())
print("Largest gap (mins):", gaps.max())

Coverage: 2024-01-01 00:00:00+00:00 to 2025-01-01 23:52:00+00:00
Median gap (mins): 52.0
Largest gap (mins): 240.0


## Match 2024 flights (ATL) to preceding weather

Longest gap of timestamps is 4 hours. We allow observations of up to 60 minutes old as an initial choice.

In [38]:
flights = pd.read_csv(PROJECT_ROOT / "data/processed/flights_2024_top10.csv.gz", usecols=["FlightDate", "Origin", "prediction_cutoff_utc"])
flights_atl = flights.loc[flights["Origin"].eq("ATL")].copy()

flights_atl["prediction_cutoff_utc"] = pd.to_datetime(flights_atl["prediction_cutoff_utc"], utc=True)

matched = pd.merge_asof(
    flights_atl.sort_values("prediction_cutoff_utc"),
    weather_clean[["weather_time_utc"] + weather_vars + ["precipitation_trace", "ceiling_unlimited"]],
    left_on="prediction_cutoff_utc",
    right_on="weather_time_utc",
    direction="backward",
    tolerance=pd.Timedelta("60min"),
)

matched["weather_age_minutes"] = (matched["prediction_cutoff_utc"] - matched["weather_time_utc"]).dt.total_seconds() / 60

ages = matched["weather_age_minutes"].dropna()
assert len(matched) == len(flights_atl)
assert ages.between(0, 60).all()

print(f"Flights: {len(matched)}")
print(f"Matched: {matched["weather_time_utc"].notna().mean():.1%}")
display(ages.describe())

Flights: 338500
Matched: 100.0%


count    338371.000000
mean         24.921920
std          16.921894
min           0.000000
25%           8.000000
50%          23.000000
75%          38.000000
max          59.000000
Name: weather_age_minutes, dtype: float64

Monthly summary for flights

In [39]:
monthly = matched.assign(
    month = pd.to_datetime(matched["FlightDate"]).dt.month,
    has_weather=matched["weather_time_utc"].notna(),
).groupby("month").agg(
    flights=("has_weather", "size"),
    matched_pct=("has_weather", "mean"),
    median_age_minutes=("weather_age_minutes", "median"),
)

monthly["matched_pct"] *= 100
display(monthly.round(2))

,flights,matched_pct,median_age_minutes
month,,,
1,26043,100.00,23.0
2,24800,100.00,23.0
3,28382,100.00,23.0
4,28362,100.00,23.0
5,29697,99.57,23.0
6,29218,100.00,24.0
7,28294,100.00,22.0
8,29175,100.00,23.0
9,28012,100.00,23.0


Check the measurements that are actually available in this test

In [40]:
features = weather_vars + ["precipitation_trace", "ceiling_unlimited"]

display(matched[features].isna().mean().mul(100).round(2).rename("missing_pct").to_frame())

,missing_pct
temperature,0.04
dew_point_temperature,0.04
wind_speed,0.04
wind_gust,87.85
visibility,0.38
ceiling_height,65.14
precipitation,31.98
precipitation_trace,31.98
ceiling_unlimited,18.11


Check ceiling categories separately

In [41]:
ceiling = matched["ceiling_unlimited"]

ceiling_status = pd.Series("unkown", index=matched.index)
ceiling_status.loc[ceiling.eq(True).fillna(False)] = "unlimited"
ceiling_status.loc[ceiling.eq(False).fillna(False)] = "reported height"

display(ceiling_status.value_counts(normalize=True).mul(100).round(2).rename("percent").to_frame())

,percent
unlimited,47.03
reported height,34.86
unkown,18.11


Check unmatched flights in May and Decemeber

In [42]:
unmatched = matched.loc[matched["weather_time_utc"].isna()]

print(f"Unmatched flights: {len(unmatched):,}")
display(unmatched.groupby("FlightDate").size().rename("unmatched_flights").to_frame())

Unmatched flights: 129


,unmatched_flights
FlightDate,
2024-05-03,129


In [43]:
last_obs = weather_clean["weather_time_utc"].max()

display(unmatched.groupby("FlightDate")["prediction_cutoff_utc"].agg(["min", "max"]))

print(f"Cutoffs beyond weather-file coverage: {unmatched["prediction_cutoff_utc"].gt(last_obs).sum()}")

,min,max
FlightDate,,
2024-05-03,2024-05-03 08:00:00+00:00,2024-05-03 10:51:00+00:00


Cutoffs beyond weather-file coverage: 0


In [44]:
display(weather_clean.loc[
    weather_clean["weather_time_utc"].between(
        "2024-05-03 06:00:00+00:00",
        "2024-05-03 12:00:00+00:00",
    ),
    ["weather_time_utc", "temperature", "wind_speed", "visibility"],
])

,weather_time_utc,temperature,wind_speed,visibility
4506,2024-05-03 06:00:00+00:00,23.3,2.6,16.000
4507,2024-05-03 06:52:00+00:00,22.8,2.1,16.093
4508,2024-05-03 10:52:00+00:00,21.1,2.1,16.093
4509,2024-05-03 11:52:00+00:00,21.1,1.5,16.093
4510,2024-05-03 12:00:00+00:00,21.1,1.5,16.000


Save pilot's audit summaries

In [45]:
WEATHER_AUDIT = PROJECT_ROOT / "outputs/weather_audit"
WEATHER_AUDIT.mkdir(parents=True, exist_ok=True)

monthly.to_csv(WEATHER_AUDIT / "atl_2024_montly_coverage.csv")

matched[features].isna().mean().mul(100).rename("missing_pct").to_csv(WEATHER_AUDIT / "atl_2024_feature_missingness.csv")

unmatched.groupby("FlightDate").size().rename("unmatched_flights").to_csv(WEATHER_AUDIT / "atl_2024_unmatched_dates.csv")